# Masked diffusion language model from scratch

A masked (absorbing state) diffusion language model following MDLM (Sahoo et al., 2024) and MD4 (Shi et al., 2024), trained on TinyStories, and a GPT style autoregressive model as the reference. Both use the same transformer, tokenizer, data, token budget and optimizer settings. Only the attention mask, the loss and the sampler differ.

`MODE` picks what a run does:
- `smoke`: tiny CPU run of every part, with the correctness checks
- `prep`: tokenizer and token arrays
- `pilot`: GPU checks, throughput benchmark and a short training run
- `train`, `eval`, `demo`

`MODEL` is `mdlm`, `ar` or `both`. With `both` and two GPUs, each model trains on its own GPU at the same time.

In [ ]:
MODE = "smoke"
MODEL = "both"

SMOKE = MODE == "smoke"
VOCAB = 1024 if SMOKE else 8192
SEQ = 64 if SMOKE else 256
DIM, LAYERS, HEADS = (128, 2, 4) if SMOKE else (512, 10, 8)
BATCH = 16 if SMOKE else 128
STEPS = {"smoke": 300, "pilot": 2000}.get(MODE, 18000)
HOURS = {"smoke": 0.2, "pilot": 0.13}.get(MODE, 4.6)
LR = 3e-3 if SMOKE else 1e-3
WARMUP = 30 if SMOKE else 500
WD, CLIP, EMA = 0.1, 1.0, 0.999
EPS = 1e-3
COMPILE = MODE in ("pilot", "train")
SEED = 0
HF_REPO = "shantanu-das/mdlm-tinystories"

In [ ]:
import os, math, time, json, glob, copy, threading, textwrap
from contextlib import nullcontext
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from huggingface_hub import hf_hub_download

KAGGLE = os.path.exists("/kaggle/working")
OUT = "/kaggle/working" if KAGGLE else ("work/smoke" if SMOKE else "work")
os.makedirs(OUT, exist_ok=True)
DEVS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
print(torch.__version__, DEVS)
AMP = None
if DEVS[0] != "cpu":
    print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
    torch.ones(8, 8, device="cuda") @ torch.ones(8, 8, device="cuda")  # fails here if this torch build has no kernels for the GPU
    AMP = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16


def autocast(dev):
    return torch.autocast("cuda", dtype=AMP) if str(dev).startswith("cuda") else nullcontext()


def find(name):
    hits = glob.glob(f"{OUT}/{name}") + sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    return hits[0] if hits else None


def get(name):
    return find(name) or hf_hub_download(HF_REPO, name)

## Data and tokenizer

TinyStoriesV2 (GPT-4 generated stories). The raw text is clean UTF-8. The only non-ASCII characters are curly quotes, a few dashes and ellipses, which are mapped to ASCII so they do not take up vocabulary.

A byte level BPE tokenizer with 8192 tokens is trained on the training split, with two special tokens: `<eos>` (id 0) and `<mask>` (id 1). Each story is followed by `<eos>` and all stories are concatenated into one uint16 stream per split. Training windows of `SEQ + 1` tokens are cut from the stream at random offsets. The smoke run uses a slice of the validation file for both splits.

In [ ]:
EOS, MASK = 0, 1
FIX = str.maketrans({"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-", "\u2026": "..."})


def read_docs(name):
    path = hf_hub_download("roneneldan/TinyStories", name, repo_type="dataset")
    text = open(path, encoding="utf-8").read().translate(FIX)
    return [d.strip() for d in text.split("<|endoftext|>") if d.strip()]


def encode(tok, docs):
    parts = []
    for i in range(0, len(docs), 50000):
        for e in tok.encode_batch(docs[i:i + 50000]):
            parts.append(np.array(e.ids + [EOS], dtype=np.uint16))
    return np.concatenate(parts)


def prep():
    val = read_docs("TinyStoriesV2-GPT4-valid.txt")
    if SMOKE:
        train, val = val[:20000], val[20000:]
    else:
        train = read_docs("TinyStoriesV2-GPT4-train.txt")
    print(len(train), "train docs,", len(val), "val docs")
    print(train[1])
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=VOCAB, special_tokens=["<eos>", "<mask>"],
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tok.train_from_iterator(train[:300000], trainer)
    tok.save(f"{OUT}/tok.json")
    for name, docs in (("train", train), ("val", val)):
        ids = encode(tok, docs)
        np.save(f"{OUT}/{name}.npy", ids)
        print(name, len(ids), "tokens")


if MODE in ("smoke", "prep"):
    prep()

In [ ]:
tok = Tokenizer.from_file(get("tok.json"))
V = tok.get_vocab_size()
PIECES = [tok.decode([i]) for i in range(V)]
PIECES[EOS], PIECES[MASK] = "\n\n", "\u2588"


def show(ids):
    return "".join(PIECES[i] for i in ids)


def get_batch(ids, B, L, gen):
    ix = torch.randint(len(ids) - L, (B,), generator=gen).tolist()
    return torch.from_numpy(np.stack([ids[i:i + L] for i in ix]).astype(np.int64))


if MODE != "demo":
    train_ids = np.load(find("train.npy"), mmap_mode="r")
    val_ids = np.load(find("val.npy"), mmap_mode="r")
    n = len(val_ids) // (SEQ + 1)
    val_windows = torch.from_numpy(np.array(val_ids[:n * (SEQ + 1)]).reshape(n, SEQ + 1).astype(np.int64))
    print(V, "vocab,", len(train_ids), "train tokens,", len(val_windows), "val windows")
    print(show(train_ids[:200].tolist()))

## Backbone

A small pre-norm transformer: RoPE, RMSNorm, SwiGLU MLP, `F.scaled_dot_product_attention`, tied input and output embeddings. The only switch between the two models is `causal`.

The diffusion model gets no time input. The number of masks in $x_t$ already tells it how far along the process is, and MDLM shows that with the SUBS parameterization time conditioning is not needed. The mask logit is set to $-\infty$ in both models, since mask never appears as a target.

The KV cache is only used for AR sampling: one forward pass over the prompt at position 0, then one token at a time.

In [ ]:
def rope(x, cos, sin):
    x1, x2 = x.float().chunk(2, -1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).type_as(x)


class Block(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        hidden = 64 * round(8 * dim / 3 / 64)
        self.n1, self.n2 = nn.RMSNorm(dim), nn.RMSNorm(dim)
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        self.up = nn.Linear(dim, 2 * hidden, bias=False)
        self.down = nn.Linear(hidden, dim, bias=False)

    def forward(self, x, cos, sin, causal, kv=None, pos=0):
        B, T, D = x.shape
        q, k, v = self.qkv(self.n1(x)).view(B, T, 3, self.heads, -1).transpose(1, 3).unbind(2)
        q, k = rope(q, cos, sin), rope(k, cos, sin)
        if kv is not None:
            kv[0][:, :, pos:pos + T] = k
            kv[1][:, :, pos:pos + T] = v
            k, v = kv[0][:, :, :pos + T], kv[1][:, :, :pos + T]
        # with a cache, T > 1 only happens for the prompt at pos 0
        y = F.scaled_dot_product_attention(q, k, v, is_causal=causal and T > 1)
        x = x + self.proj(y.transpose(1, 2).reshape(B, T, D))
        a, b = self.up(self.n2(x)).chunk(2, -1)
        return x + self.down(F.silu(a) * b)


class Transformer(nn.Module):
    def __init__(self, vocab, dim, layers, heads, seq, causal):
        super().__init__()
        self.cfg = dict(vocab=vocab, dim=dim, layers=layers, heads=heads, seq=seq, causal=causal)
        self.causal = causal
        self.emb = nn.Embedding(vocab, dim)
        self.blocks = nn.ModuleList(Block(dim, heads) for _ in range(layers))
        self.norm = nn.RMSNorm(dim)
        hd = dim // heads
        a = torch.outer(torch.arange(seq).float(), 10000 ** (-torch.arange(0, hd, 2).float() / hd))
        self.register_buffer("cos", a.cos(), persistent=False)
        self.register_buffer("sin", a.sin(), persistent=False)
        for name, p in self.named_parameters():
            if p.dim() == 2:
                std = 0.02 / math.sqrt(2 * layers) if name.endswith(("proj.weight", "down.weight")) else 0.02
                nn.init.normal_(p, std=std)

    def forward(self, idx, cache=None):
        T = idx.shape[1]
        pos = cache["pos"] if cache is not None else 0
        cos, sin = self.cos[pos:pos + T], self.sin[pos:pos + T]
        x = self.emb(idx)
        for i, blk in enumerate(self.blocks):
            x = blk(x, cos, sin, self.causal, cache["kv"][i] if cache is not None else None, pos)
        if cache is not None:
            cache["pos"] += T
        logits = F.linear(self.norm(x), self.emb.weight)
        logits[..., MASK] = float("-inf")  # never predict mask
        return logits

    def new_cache(self, B, n, dtype):
        c = self.cfg
        shape = (B, c["heads"], n, c["dim"] // c["heads"])
        dev = self.emb.weight.device
        return {"pos": 0, "kv": [(torch.zeros(shape, dtype=dtype, device=dev), torch.zeros(shape, dtype=dtype, device=dev))
                                 for _ in self.blocks]}


def n_params(m):
    return sum(p.numel() for p in m.parameters()) - m.emb.weight.numel()

## Masked diffusion: forward process and loss

With the linear schedule $\alpha_t = 1 - t$, each token is replaced by the mask token $m$ independently with probability $t$:

$$q(x_t^i \mid x_0^i) = \alpha_t \, \delta_{x_0^i} + (1 - \alpha_t) \, \delta_{m}$$

The model reads $x_t$ and outputs $p_\theta(x_0^i \mid x_t)$ at every position. Under SUBS, unmasked positions are copied through and carry no loss, so only masked positions count. The continuous time NELBO is

$$\mathcal{L} = \mathbb{E}_{t \sim U(0,1)} \, \mathbb{E}_{q(x_t \mid x_0)} \Big[ \frac{-\alpha_t'}{1 - \alpha_t} \sum_{i : x_t^i = m} -\log p_\theta(x_0^i \mid x_t) \Big], \qquad \frac{-\alpha_t'}{1 - \alpha_t} = \frac{1}{t}$$

Dividing by the sequence length $L$ gives a per token bound on the negative log likelihood. $t$ is drawn from $[\epsilon, 1]$ with $\epsilon = 10^{-3}$, stratified across the batch: one uniform offset $u_0$, then $t_b = \epsilon + (1 - \epsilon)((u_0 + b/B) \bmod 1)$. This lowers the variance of the loss.

In [ ]:
def q_sample(x0, t, gen=None):
    m = torch.rand(x0.shape, device=x0.device, generator=gen) < t[:, None]
    return torch.where(m, MASK, x0), m


def mdlm_loss(model, x0, gen=None):
    B, L = x0.shape
    u = (torch.rand(1, device=x0.device, generator=gen) + torch.arange(B, device=x0.device) / B) % 1
    t = EPS + (1 - EPS) * u
    xt, m = q_sample(x0, t, gen)
    nll = F.cross_entropy(model(xt).float().flatten(0, 1), x0.flatten(), reduction="none").view(B, L)
    return (nll * m).sum(1) / (t * L)

## Masked diffusion: reverse process

For $s < t$ the true posterior keeps every unmasked token, and unmasks each masked token with probability

$$\frac{\alpha_s - \alpha_t}{1 - \alpha_t} = \frac{t - s}{t},$$

setting it to $x_0^i$. The ancestral sampler uses the same step with $x_0^i$ drawn from $p_\theta(x_0^i \mid x_t)$. It starts from all masks at $t = 1$ and walks a uniform grid down to $0$. At $s = 0$ the probability is 1, so nothing is left masked.

Details:
- Categorical draws use Gumbel max in float64. Zheng et al. (2024) showed that float32 Gumbel noise lowers the effective temperature, which makes generative perplexity look better than it is.
- There is no time input, so the logits only change when $x_t$ changes. Rows where nothing was unmasked in the last step reuse their cached logits. The sampler returns the number of forward passes per sample it actually ran.
- Positions that start unmasked are never touched. That gives prefix prompts and infilling (fixed prefix and suffix) with no extra training.
- Samples start with `<eos>` at position 0 for both models. Training windows start at random offsets in the stream, so without it the samples would start in the middle of a story.

In [ ]:
def gumbel_argmax(logits, gen=None):
    u = torch.rand(logits.shape, dtype=torch.float64, device=logits.device, generator=gen)
    return (logits.double() - torch.log(-torch.log(u))).argmax(-1)


@torch.no_grad()
def mdlm_sample(model, x, steps, gen=None, trace=None):
    x = x.clone()
    B, L = x.shape
    ts = torch.linspace(1, 0, steps + 1, dtype=torch.float64).tolist()
    logits = torch.zeros(B, L, model.cfg["vocab"], device=x.device)
    stale = (x == MASK).any(1)
    nfe = 0
    if trace is not None:
        trace.append(x.cpu().clone())
    for t, s in zip(ts[:-1], ts[1:]):
        if stale.any():
            with autocast(x.device):
                logits[stale] = model(x[stale]).float()
            nfe += stale.sum().item()
        u = torch.rand(B, L, dtype=torch.float64, device=x.device, generator=gen)
        reveal = (x == MASK) & (u < (t - s) / t)
        x[reveal] = gumbel_argmax(logits[reveal], gen)
        stale = reveal.any(1) & (x == MASK).any(1)
        if trace is not None:
            trace.append(x.cpu().clone())
    return x, nfe / B

## Autoregressive baseline

Same `Transformer` with `causal=True` and next token cross entropy on the same windows. Sampling uses the KV cache and the same float64 Gumbel draw at temperature 1.

In [ ]:
def ar_loss(model, x):
    logits = model(x[:, :-1])
    return F.cross_entropy(logits.float().flatten(0, 1), x[:, 1:].flatten(), reduction="none").view(len(x), -1).mean(1)


@torch.no_grad()
def ar_sample(model, prompt, n, gen=None):
    dtype = AMP if prompt.is_cuda else torch.float32
    cache = model.new_cache(prompt.shape[0], prompt.shape[1] + n, dtype)
    out = [prompt]
    with autocast(prompt.device):
        logits = model(prompt, cache)[:, -1]
        for i in range(n):
            out.append(gumbel_argmax(logits, gen)[:, None])
            if i < n - 1:
                logits = model(out[-1], cache)[:, -1]
    return torch.cat(out, 1), n


def loss_fn(kind, model, x):
    return mdlm_loss(model, x[:, :-1]) if kind == "mdlm" else ar_loss(model, x)

## Correctness checks

Run in the smoke and pilot runs, before any long training:
- the forward process masks a fraction $t$ of tokens
- at init, the loss per masked token at $t = 1$ is close to $\log V$
- the sampler ends with no masks, never changes an unmasked token, and unmasks about $L/T$ tokens per step
- AR logits with the KV cache match a full forward pass
- both models overfit a tiny batch (4 sequences) and reproduce it when sampling from a 4 token prefix (smoke only)

The overfit check repeats the 4 sequences 8 times per step. With only 4 draws of $t$ per step, the $1/t$ weighted loss is noisy enough that the diffusion model stalls well above its floor of $\log 4 / L$ per token. It also has to work out absolute positions from whichever tokens are visible, so it memorizes more slowly than AR. Sampling is at temperature 1, so even at a loss near zero a sample can pick a low probability token and drift from there, and the check allows up to 10% wrong tokens.

In [ ]:
def run_checks(dev):
    g = torch.Generator(dev).manual_seed(0)
    x0 = torch.randint(2, V, (128, SEQ), device=dev, generator=g)
    for t in (0.1, 0.5, 0.9):
        _, m = q_sample(x0, torch.full((128,), t, device=dev), g)
        assert abs(m.float().mean().item() - t) < 0.01

    model = Transformer(V, DIM, LAYERS, HEADS, SEQ, causal=False).to(dev)
    with torch.no_grad():
        nll = F.cross_entropy(model(torch.full_like(x0[:16], MASK)).flatten(0, 1), x0[:16].flatten()).item()
    print("init loss at t=1:", round(nll, 3), "log V:", round(math.log(V), 3))
    assert abs(nll / math.log(V) - 1) < 0.05

    T = 16
    x = torch.full((128, SEQ), MASK, device=dev)
    x[:, 0] = EOS
    states = []
    mdlm_sample(model, x, T, g, states)
    S = torch.stack(states)
    assert (S[-1] != MASK).all()
    was = S[:-1] != MASK
    assert (S[1:][was] == S[:-1][was]).all()
    per_step = ((S[:-1] == MASK) & (S[1:] != MASK)).sum(2).float().mean(1)
    print("unmasked per step:", [round(v, 1) for v in per_step.tolist()], "expected", round((SEQ - 1) / T, 1))
    assert ((per_step / ((SEQ - 1) / T) - 1).abs() < 0.15).all()

    ar = Transformer(V, DIM, LAYERS, HEADS, SEQ, causal=True).to(dev)
    x = torch.randint(2, V, (2, 20), device=dev, generator=g)
    with torch.no_grad():
        full = ar(x)
        cache = ar.new_cache(2, 20, torch.float32)
        inc = torch.cat([ar(x[:, :10], cache)] + [ar(x[:, i:i + 1], cache) for i in range(10, 20)], 1)
    torch.testing.assert_close(full, inc, atol=1e-3, rtol=1e-3)
    print("checks passed")


def overfit(kind, dev, steps=2000):
    torch.manual_seed(0)
    model = Transformer(V, DIM, LAYERS, HEADS, SEQ, causal=kind == "ar").to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    x = get_batch(train_ids, 4, SEQ + 1, torch.Generator().manual_seed(1)).to(dev)
    losses = []
    for i in range(steps):
        for g in opt.param_groups:
            g["lr"] = 1e-3 * (1 - i / steps)
        loss = loss_fn(kind, model, x.repeat(8, 1)).mean()  # 8 draws of t per sequence per step
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        opt.step()
        losses.append(loss.item())
    gen = torch.Generator(dev).manual_seed(0)
    if kind == "ar":
        out, _ = ar_sample(model, x[:, :4], SEQ - 4, gen)
    else:
        c = x[:, :SEQ].clone()
        c[:, 4:] = MASK
        out, _ = mdlm_sample(model, c, SEQ, gen)
    loss, acc = np.mean(losses[-100:]), (out == x[:, :SEQ]).float().mean().item()
    print(kind, "overfit loss", round(loss, 4), "tokens reproduced", acc)
    assert loss < 0.2 and acc > 0.9


if MODE in ("smoke", "pilot"):
    run_checks(DEVS[0])
if SMOKE:
    overfit("mdlm", "cpu")
    overfit("ar", "cpu")

## Training

Identical for both models: AdamW (betas 0.9, 0.95, weight decay 0.1 on matrices), linear warmup then cosine decay to 10% of the peak learning rate, gradient clipping at 1.0, an EMA of the weights (decay 0.999) used for evaluation and sampling, fp16 autocast with a gradient scaler on T4. Both models draw the same batches from the same seed, so they see the same tokens. A checkpoint is written every 30 minutes and training stops at `STEPS` or after `HOURS`, whichever comes first. A `train` run resumes from an existing checkpoint.

In [ ]:
def lr_at(i):
    if i < WARMUP:
        return LR * (i + 1) / WARMUP
    p = min(1.0, (i - WARMUP) / max(1, STEPS - WARMUP))
    return LR * (0.1 + 0.45 * (1 + math.cos(math.pi * p)))


def make_opt(model, dev):
    groups = [{"params": [p for p in model.parameters() if p.dim() >= 2], "weight_decay": WD},
              {"params": [p for p in model.parameters() if p.dim() < 2], "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=LR, betas=(0.9, 0.95), fused=dev.startswith("cuda"))


def train_step(kind, fwd, model, opt, scaler, x, dev):
    with autocast(dev):
        loss = loss_fn(kind, fwd, x).mean()
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    nn.utils.clip_grad_norm_(model.parameters(), CLIP)
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    return loss.detach()


def save(kind, model, ema, opt, scaler, step, log):
    torch.save({"model": model.state_dict(), "ema": ema.state_dict(), "opt": opt.state_dict(),
                "scaler": scaler.state_dict(), "step": step, "log": log}, f"{OUT}/{kind}.pt")
    torch.save({"cfg": ema.cfg, "model": {k: v.half() for k, v in ema.state_dict().items()}}, f"{OUT}/{kind}_ema.pt")
    json.dump(log, open(f"{OUT}/{kind}_log.json", "w"))


COMPILE_LOCK = threading.Lock()


def train(kind, dev):
    torch.manual_seed(SEED)
    model = Transformer(V, DIM, LAYERS, HEADS, SEQ, causal=kind == "ar").to(dev)
    ema = copy.deepcopy(model).requires_grad_(False)
    opt = make_opt(model, dev)
    scaler = torch.amp.GradScaler(enabled=AMP == torch.float16 and dev.startswith("cuda"))
    step, log = 0, []
    ck = find(f"{kind}.pt") if MODE == "train" else None
    if ck:
        s = torch.load(ck, map_location=dev)
        model.load_state_dict(s["model"])
        ema.load_state_dict(s["ema"])
        opt.load_state_dict(s["opt"])
        scaler.load_state_dict(s["scaler"])
        step, log = s["step"], s["log"]
        print(kind, "resumed at step", step)
    print(kind, dev, f"{n_params(model) / 1e6:.1f}M non-embedding params")
    fwd = torch.compile(model) if COMPILE else model
    gen = torch.Generator().manual_seed(SEED + step)
    t0, start, last_save, losses = time.time(), step, time.time(), []
    while step < STEPS and time.time() - t0 < HOURS * 3600:
        for g in opt.param_groups:
            g["lr"] = lr_at(step)
        x = get_batch(train_ids, BATCH, SEQ + 1, gen).pin_memory().to(dev, non_blocking=True) if dev != "cpu" \
            else get_batch(train_ids, BATCH, SEQ + 1, gen)
        # dynamo cannot compile in two threads at once, so the first steps (where it compiles) take turns
        with COMPILE_LOCK if step - start < 3 else nullcontext():
            losses.append(train_step(kind, fwd, model, opt, scaler, x, dev))
        with torch.no_grad():
            for e, p in zip(ema.parameters(), model.parameters()):
                e.lerp_(p, 1 - EMA)
        step += 1
        if step % 100 == 0 or step == STEPS:
            loss, el = torch.stack(losses).mean().item(), time.time() - t0
            losses = []
            log.append((step, loss))
            print(f"{kind} step {step} loss {loss:.3f} lr {lr_at(step):.1e} "
                  f"{(step - start) * BATCH * SEQ / el:.0f} tok/s {el / 60:.1f} min", flush=True)
        if time.time() - last_save > 1800:
            save(kind, model, ema, opt, scaler, step, log)
            last_save = time.time()
    save(kind, model, ema, opt, scaler, step, log)
    print(kind, "done at step", step, f"{step * BATCH * SEQ / 1e6:.0f}M tokens")


def bench(kind, dev, dim, layers, heads, comp, n=20):
    model = Transformer(V, dim, layers, heads, SEQ, causal=kind == "ar").to(dev)
    opt, scaler = make_opt(model, dev), torch.amp.GradScaler(enabled=AMP == torch.float16)
    fwd = torch.compile(model) if comp else model
    gen = torch.Generator().manual_seed(0)
    for i in range(n + 5):
        if i == 5:
            torch.cuda.synchronize(dev)
            t0 = time.time()
        train_step(kind, fwd, model, opt, scaler, get_batch(train_ids, BATCH, SEQ + 1, gen).to(dev), dev)
    torch.cuda.synchronize(dev)
    rate = n * BATCH * SEQ / (time.time() - t0)
    print(kind, dim, layers, heads, "compile" if comp else "eager", f"{n_params(model) / 1e6:.1f}M",
          f"{rate:.0f} tok/s", f"{torch.cuda.max_memory_allocated(dev) / 1e9:.1f} GB")
    del model, opt, fwd
    torch.cuda.empty_cache()


if MODE == "pilot":
    for dim, layers, heads in [(512, 8, 8), (512, 10, 8), (640, 10, 10)]:
        bench("mdlm", DEVS[0], dim, layers, heads, False)
    for kind in ("mdlm", "ar"):
        bench(kind, DEVS[0], DIM, LAYERS, HEADS, True)

if MODE in ("smoke", "pilot", "train"):
    kinds = ["mdlm", "ar"] if MODEL == "both" else [MODEL]
    if len(kinds) > 1 and len(DEVS) >= len(kinds):
        threads = [threading.Thread(target=train, args=(k, d)) for k, d in zip(kinds, DEVS)]
        for th in threads:
            th.start()
        for th in threads:
            th.join()
    else:
        for k in kinds:
            train(k, DEVS[0])

## Evaluation

1. Validation perplexity on the official validation split, cut into non-overlapping windows. For AR it is exact. For diffusion it is the exponentiated NELBO, an upper bound on perplexity, averaged over 4 draws of $t$ and the mask per window with a fixed seed.
2. Generative perplexity: 256 unconditional samples per setting, cut at the first `<eos>`, scored by GPT-2 Large, which is a fixed judge and is never trained. Unigram entropy (over our token ids, per sample, averaged) catches repetitive text. Diffusion is run at several step counts. The first 256 whole validation stories, capped at 255 tokens like the samples, are scored the same way as a reference. Samples that end at once are empty and are left out of perplexity and entropy.
3. Seconds per sample (batch 1) and forward passes per sample.
4. The first three samples of each setting, uncurated, from a fixed seed.

In [ ]:
def load(kind, dev):
    s = torch.load(get(f"{kind}_ema.pt"), map_location=dev)
    model = Transformer(**s["cfg"]).to(dev)
    model.load_state_dict({k: v.float() for k, v in s["model"].items()})
    return model.eval()


@torch.no_grad()
def val_nll(kind, model, dev, K=4):
    gen = torch.Generator(dev).manual_seed(SEED)
    windows = val_windows[:512] if SMOKE else val_windows
    tot = 0.0
    for i in range(0, len(windows), 64):
        x = windows[i:i + 64].to(dev)
        with autocast(dev):
            if kind == "ar":
                l = ar_loss(model, x)
            else:
                l = torch.stack([mdlm_loss(model, x[:, :-1], gen) for _ in range(K)]).mean(0)
        tot += l.sum().item()
    return tot / len(windows)


def unconditional(kind, model, n, steps, dev, seed, bs=64):
    gen = torch.Generator(dev).manual_seed(seed)
    out, nfe = [], 0
    for i in range(0, n, bs):
        b = min(bs, n - i)
        if kind == "ar":
            x, f = ar_sample(model, torch.full((b, 1), EOS, device=dev), SEQ - 1, gen)
        else:
            x = torch.full((b, SEQ), MASK, device=dev)
            x[:, 0] = EOS
            x, f = mdlm_sample(model, x, steps, gen)
        out += x.tolist()
        nfe += f * b
    return out, nfe / n


def story(ids):
    ids = list(ids[1:])
    return ids[:ids.index(EOS)] if EOS in ids else ids


def judge_ppl(texts, jt, jm, dev):
    nll, n = 0.0, 0
    for text in texts:
        ids = torch.tensor([[jt.bos_token_id] + jt.encode(text)[:1000]], device=dev)
        with torch.no_grad():
            logits = jm(ids).logits.float()
        nll += F.cross_entropy(logits[0, :-1], ids[0, 1:], reduction="sum").item()
        n += ids.shape[1] - 1
    return math.exp(nll / n)


def entropy(ids):
    _, c = np.unique(ids, return_counts=True)
    p = c / c.sum()
    return float(-(p * np.log(p)).sum())


def latency(kind, model, steps, dev, n=3):
    times = []
    for i in range(n + 1):
        if dev != "cpu":
            torch.cuda.synchronize(dev)
        t0 = time.time()
        _, nfe = unconditional(kind, model, 1, steps, dev, seed=100 + i)
        if dev != "cpu":
            torch.cuda.synchronize(dev)
        if i:
            times.append(time.time() - t0)
    return float(np.mean(times)), nfe


if MODE in ("smoke", "eval"):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    dev = DEVS[0]
    nets = {k: load(k, dev) for k in ("mdlm", "ar")}
    res = {"val": {}, "gen": [], "samples": {}}
    for k, m in nets.items():
        nll = val_nll(k, m, dev)
        res["val"][k] = {"nll": nll, "ppl": math.exp(nll), "bound": k == "mdlm"}
        print(k, "val nll", round(nll, 4), "ppl", round(math.exp(nll), 3), "(upper bound)" if k == "mdlm" else "(exact)")

    jname = "sshleifer/tiny-gpt2" if SMOKE else "openai-community/gpt2-large"
    jt = AutoTokenizer.from_pretrained(jname)
    jm = AutoModelForCausalLM.from_pretrained(jname).to(dev).eval()
    if dev != "cpu":
        jm.half()
    N = 8 if SMOKE else 256
    head = np.array(val_ids[:2 * N * SEQ]).astype(np.int64)
    ends = np.flatnonzero(head == EOS)
    ref = [head[a + 1:b][:SEQ - 1] for a, b in zip(np.r_[-1, ends[:-1]], ends)][:N]
    res["gen"].append({"model": "data", "steps": None, "gen_ppl": judge_ppl([tok.decode(s.tolist()) for s in ref], jt, jm, dev),
                       "entropy": float(np.mean([entropy(s) for s in ref])), "length": float(np.mean([len(s) for s in ref]))})
    print(res["gen"][-1], flush=True)
    settings = [("ar", SEQ - 1)] + [("mdlm", s) for s in ([4, 16] if SMOKE else [16, 32, 64, 128, 256, 1024])]
    for kind, steps in settings:
        ids, nfe = unconditional(kind, nets[kind], N, steps, dev, seed=SEED)
        st = [story(x) for x in ids]
        texts = [tok.decode(s) for s in st]
        sec, nfe1 = latency(kind, nets[kind], steps, dev)
        row = {"model": kind, "steps": steps, "gen_ppl": judge_ppl([t for t in texts if t], jt, jm, dev),
               "entropy": float(np.mean([entropy(s) for s in st if s])), "length": float(np.mean([len(s) for s in st])),
               "nfe": nfe, "sec": sec}
        res["gen"].append(row)
        res["samples"][f"{kind}_{steps}"] = texts[:3]
        print(row, flush=True)
    json.dump(res, open(f"{OUT}/results.json", "w"), indent=1)

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for k, label in (("mdlm", "MDLM (NELBO bound)"), ("ar", "AR (exact NLL)")):
        log = np.array(json.load(open(get(f"{k}_log.json"))))
        log = log[log[:, 0] >= WARMUP]
        ax[0].plot(log[:, 0] * BATCH * SEQ / 1e6, log[:, 1], label=label)
    ax[0].set(xlabel="training tokens (M)", ylabel="train loss per token (nats)")
    ax[0].legend()
    m = [r for r in res["gen"] if r["model"] == "mdlm"]
    ax[1].plot([r["steps"] for r in m], [r["gen_ppl"] for r in m], "o-", label="MDLM, ancestral sampler")
    for r, style, label in ((res["gen"][1], "--", "AR, 255 tokens"), (res["gen"][0], ":", "validation stories")):
        ax[1].axhline(r["gen_ppl"], ls=style, color="k" if r["model"] == "data" else "C1", label=label)
    ax[1].set(xscale="log", xlabel="sampling steps", ylabel="GPT-2 Large perplexity")
    ax[1].minorticks_off()
    ax[1].set_xticks([r["steps"] for r in m], [r["steps"] for r in m])
    ax[1].legend()
    fig.tight_layout()
    fig.savefig(f"{OUT}/results.png", dpi=130)
    plt.show()

## Denoising view

One unconditional sample traced step by step. Masks are drawn as a block character. Only the first story (up to the first generated `<eos>`) is shown.

In [ ]:
def denoise_states(model, steps, seed, dev):
    x = torch.full((1, SEQ), MASK, device=dev)
    x[:, 0] = EOS
    states = []
    mdlm_sample(model, x, steps, torch.Generator(dev).manual_seed(seed), states)
    states = [s[0].tolist() for s in states]
    end = len(story(states[-1])) + 1
    return [s[1:end] for s in states]


def denoise_figure(states, path, k=6):
    picks = np.linspace(0, len(states) - 1, k).astype(int)
    fig, axes = plt.subplots(k, 1, figsize=(10, 1.5 * k))
    for ax, i in zip(axes, picks):
        ax.axis("off")
        ax.set_title(f"step {i} of {len(states) - 1}", loc="left", fontsize=9)
        ax.text(0, 1, textwrap.fill(show(states[i]), 105), family="monospace", fontsize=7.5, va="top")
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.show()


def denoise_gif(states, path):
    font = ImageFont.truetype(matplotlib.font_manager.findfont("DejaVu Sans Mono"), 15)
    texts = [textwrap.fill(show(s), 90) for s in states]
    h = 30 + 19 * max(t.count("\n") + 1 for t in texts)
    frames = []
    for t in texts:
        img = Image.new("RGB", (860, h), "white")
        ImageDraw.Draw(img).multiline_text((15, 15), t, fill="black", font=font)
        frames.append(img)
    frames += [frames[-1]] * 20
    frames[0].save(path, save_all=True, append_images=frames[1:], duration=90, loop=0)


if MODE in ("smoke", "eval"):
    states = denoise_states(nets["mdlm"], SEQ // 4, 1, dev)
    denoise_figure(states, f"{OUT}/denoise.png")
    denoise_gif(states, f"{OUT}/denoise.gif")

## Demo

Gradio app for the diffusion model that runs on CPU. In `demo` mode the weights and tokenizer are downloaded from the Hugging Face model repo if they are not found locally. Give a prefix to continue it, or a prefix and a suffix to fill in the middle. The output streams every step of the reverse process.

In [ ]:
def make_demo(model):
    import gradio as gr
    L = model.cfg["seq"]

    def run(prefix, suffix, n, steps, seed):
        p, s = tok.encode(prefix).ids[:100], tok.encode(suffix).ids[:100]
        x = [EOS] + p + ([MASK] * int(n) + s + [EOS] if s else [])
        x = torch.tensor([x + [MASK] * (L - len(x))])
        stop = len(p) + int(n) + len(s) + 1 if s else None

        def text(st):
            ids = st[0, 1:stop].tolist()
            if stop is None and EOS in ids[len(p):]:
                ids = ids[:len(p) + ids[len(p):].index(EOS)]
            return show(ids)

        # sample in a thread and stream the latest state while it runs
        states, res = [], {}
        gen = torch.Generator().manual_seed(int(seed))
        th = threading.Thread(target=lambda: res.update(nfe=mdlm_sample(model, x, int(steps), gen, states)[1]))
        th.start()
        shown = 0
        while th.is_alive():
            if len(states) > shown:
                shown = len(states)
                yield text(states[-1])
            time.sleep(0.05)
        yield text(states[-1]) + f"\n\n[{steps} steps, {res['nfe']:.0f} forward passes]"

    with gr.Blocks(title="Masked diffusion LM") as demo:
        gr.Markdown("Masked diffusion language model trained on TinyStories. Masks are shown as \u2588.")
        prefix = gr.Textbox(label="Prefix", value="Once upon a time")
        suffix = gr.Textbox(label="Suffix (optional, for infilling)")
        n = gr.Slider(8, 128, 48, step=1, label="Tokens to fill between prefix and suffix")
        steps = gr.Slider(8, 256, 64, step=8, label="Sampling steps")
        seed = gr.Number(0, precision=0, label="Seed")
        btn = gr.Button("Generate")
        out = gr.Textbox(label="Sample", lines=14)
        btn.click(run, [prefix, suffix, n, steps, seed], out)
    return demo, run


if MODE in ("smoke", "demo"):
    demo, run = make_demo(load("mdlm", "cpu"))
    print(list(run("Once upon a time", "", 8, 8, 0))[-1])
    print(list(run("Tom had a red ball.", "They played all day.", 12, 8, 0))[-1])
    if MODE == "demo":
        demo.launch()